In [1]:
import smolagents
from playpen.agents import ClemAgent, ClemObservation
from clemcore.backends import load_model
from clemcore.clemgame import env, episode_results_folder_callbacks
from clemcore.backends import ModelRegistry, KeyRegistry


.--------------..--------------..--------------..--------------..--------------..--------------..--------------.
|   ______     ||   _____      ||      __      ||  ____  ____  ||   ______     ||  _________   || ____  _____  |
|  |_   __ \   ||  |_   _|     ||     /  \     || |_  _||_  _| ||  |_   __ \   || |_   ___  |  |||_   \|_   _| |
|    | |__) |  ||    | |       ||    / /\ \    ||   \ \  / /   ||    | |__) |  ||   | |_  \_|  ||  |   \ | |   |
|    |  ___/   ||    | |   _   ||   / ____ \   ||    \ \/ /    ||    |  ___/   ||   |  _|  _   ||  | |\ \| |   |
|   _| |_      ||   _| |__/ |  || _/ /    \ \_ ||    _|  |_    ||   _| |_      ||  _| |___/ |  || _| |_\   |_  |
|  |_____|     ||  |________|  |||____|  |____|||   |______|   ||  |_____|     || |_________|  |||_____|\____| |
'--------------''--------------''--------------''--------------''--------------''--------------''--------------'



In [ ]:
import os
import json

CLEMBENCH_HOME = r"C:\Users\white\Desktop\MSc_thesis\libs\playpen\clembench"
GAME_NAME = "taboo"
os.environ["CLEMBENCH_HOME"] = CLEMBENCH_HOME
os.chdir("/home/users/mmanina/project/mmanina/MSc_thesis/libs/playpen")

In [ ]:
from playpen.agents import ClemAgent, ClemObservation
from clemcore.backends import load_model
from pathlib import Path

FEEDBACK_PROMPT = Path("feedback_prompt.txt").read_text(encoding="utf-8")

REFINEMENT_PROMPT = Path("refinement_prompt.txt").read_text(encoding="utf-8")

import re

#TODO: metrics!!!
# TODO: agent traces! Maybe in clem format?

def parse_verdict(fb: str) -> dict:
    """
    Parses the feedback scores across three domains. This one is purely for logging,
    the model gets raw feedback text.
    """

    def get(label):
        m = re.search(rf"^{label}:\s*\"?(ok|problem)\"?", fb, re.I | re.M)
        return m.group(1).lower() if m else None
    return {"format":   get("FORMAT"),
            "rules":    get("RULES"),
            "strategy": get("STRATEGY"),
            "raw":      fb}

class SelfRefineAgent(ClemAgent):
    """
    An enhanced ClemAgent which emits the game responses according to the Self-Refine
    framework:
    First, a so-called draft is generated (in the same way that an ordinary Player
    response is generated). Then, the same model is queried to give feedback on this
    draft. Finally, based on the feedback, the answer is refined one final time and
    passed on to the Game Master.
    """

    def __init__(self):
        super().__init__()
        self.model = load_model("Qwen3.5-4B", gen_args=dict(temperature=0.0, max_tokens=600)) 
        self.verdicts = []
    
    def feedback(self, feedback_prompt, initial_draft) -> str:
        """
        Using the same player model, generate feedback on the initial answer,
        evaluating it, to be further used for self-refinement
        """
        replaced = feedback_prompt.format(transcript = self.history,draft = initial_draft)
        # Original Self-Feedback equation: f_bt =M(p_fb ∥ x ∥ yt)
        feedback_msgs =  [      
             {"role": "user",      "content": replaced}       
                        ]
                        
        first, second , feedback = self.model.generate_response(feedback_msgs)   # first = the final prompt that the model sees
                                                                                 # second = the raw response
        self.verdicts.append(parse_verdict(feedback))
        return feedback

    def refine(self, refinement_prompt, current_feedback, first_draft):
        """
        Using the same player model, refine the answer based on the feedback
        given during the same round
        """
        replaced = refinement_prompt.format(transcript = self.history,draft = first_draft, feedback = current_feedback)
        # Original Self-Refine equation:
        #   y1 = M (p_refine ∥ x ∥ y0 ∥ f_b0)
        
        refine_msgs =  [                                   
                     {"role": "user", "content": replaced }       
                     ]

        prompt_obj, _, final_answer = self.model.generate_response(refine_msgs)
        print(prompt_obj["inputs"])

        return final_answer


    def act(self, last: ClemObservation) -> str:

        # Generate first answer draft
        # Original equation from the Self-Refine paper: y0 = M(pgen ∥ x)
        # Here, pgen and x are both provided in the initial GM prompt
        # e.g., pgen = "...Use the following output format:..."
        # x = "CLUE: virile"
        _, _, draft = self.model.generate_response(self.history)


        feedback_result = self.feedback(FEEDBACK_PROMPT, draft)
        refined = self.refine(REFINEMENT_PROMPT, feedback_result, draft)
        self.observe(dict(role="assistant", content=refined))           # QUESTION: DO I OBSERVE THE DRAFT? gotta create some clem-like trace

        return refined

        #refinement_result = refine(self, REFINEMENT_PROMPT, feedback_result, draft, x)

        #return refinement_result



guesser = SelfRefineAgent()
describer = SelfRefineAgent()

2026-10-01 16:57:29,498 - clemcore.backends - INFO - Found registered model spec that unifies with {"model_name":"clp-chat"} -> {'model_name': 'clp-chat', 'backend': 'openai_compatible', 'context_size': '32k', 'languages': ['en', 'zh'], 'license': {'name': 'Meta', 'url': 'https://www.apache.org/licenses/LICENSE-2.0'}, 'lookup_source': 'C:\\Users\\white\\Desktop\\agents_experiments\\notebooks\\model_registry.json', 'model_config': {'multimodality': {'single_image': True, 'multiple_images': True, 'audio': False, 'video': False}}, 'model_id': 'Qwen/Qwen3-VL-30B-A3B-Instruct-FP8', 'open_weight': True, 'parameters': '30B', 'release_date': '2024-04-18', 'verify_ssl': False}
2026-10-01 16:57:29,505 - clemcore.backends - INFO - Found registry entry for backend openai_compatible -> {'backend': 'openai_compatible', 'file_name': 'openai_compatible_api.py', 'file_path': 'C:\\Users\\white\\Desktop\\MSc_thesis\\libs\\clemcore_with_changes\\clemcore\\backends\\openai_compatible_api.py', 'lookup_sourc

In [4]:
from clemcore.clemgame import env, episode_results_folder_callbacks

# Create callbacks to record the interactions in a folder; here we name the folder after the models the agent uses
callbacks = episode_results_folder_callbacks(run_dir="clp-chat", result_dir_path="playpen-records", player_model_infos="MyAgenticDescriber")

game_env = env(
    "taboo",
    instances_filter=lambda sample: sample["game_instance"].get("game_id") in (1, 5, 10),
    single_pass=False,
    callbacks=callbacks
)
game_env.reset()
agent_mapping = {"player_0": describer, "player_1": guesser}

2026-10-01 16:57:31,476 - clemcore.cli - INFO - Found '1' game matching the game_selector="taboo"
2026-10-01 16:57:31,477 - clemcore.cli - INFO - {
  "game_name": "taboo",
  "description": "Taboo game between two agents where one has to describe a word for the other to guess.",
  "main_game": "taboo",
  "players": 2,
  "image": "none",
  "languages": [
    "en"
  ],
  "benchmark": [
    "0.9",
    "1.0",
    "1.5",
    "2.0",
    "3.0"
  ],
  "regression": "small",
  "roles": [
    "Describer",
    "Guesser"
  ],
  "game_path": "C:\\Users\\white\\Desktop\\MSc_thesis\\libs\\playpen\\clembench\\taboo"
}
2026-10-01 16:57:31,480 - clemcore.run - INFO - Loading game benchmark for taboo
2026-10-01 16:57:31,824 - clemcore.run - INFO - Loading game benchmark for taboo took: 0:00:00.342806
2026-10-01 16:57:31,845 - clemcore.run - INFO - Iterating over taboo: 9 rows from 3 experiments: ['low_en', 'medium_en', 'high_en']
2026-10-01 16:57:31,847 - clemcore.run - INFO - Detected single_pass=False, 

In [ ]:
# # Now we can iterate through the agent turns, one step at a time (calling this cell multiple times)

collected = 0
episodes = 0
while  episodes < 10:
    game_env.reset()
    for agent_id in game_env.agent_iter():
        context, reward, termination, truncation, info = game_env.last()
        response = None if (termination or truncation) else agent_mapping[agent_id](context)
        game_env.step(response)
    describer.reset()
    guesser.reset()
    episodes += 1
    collected = len(describer.verdicts)
    print(f"episode {episodes}: verdicts collected: {collected}")
#     agent_id = next(iter(game_env.agent_iter()))
#     print("Agent_id:", agent_id)
#     context, reward, termination, truncation, info = game_env.last()
#     print("Reward:", reward, "Termination:", termination, "Truncation:", truncation, "Info:", info)
#     if termination or truncation:
#         response = None # we step one more time to remove the agent from the env (final reward was observed in last)
#     else:
#         response = agent_mapping[agent_id](context)
#     print("Context:", context)
#     print("Response:", response)
#     game_env.step(response)
# except StopIteration:
#     print("Episode finished.")
#     describer.reset()

Agent_id: player_0
Reward: 0.0 Termination: False Truncation: False Info: {}
2026-10-01 16:57:34,813 - clemcore.backends.openai_api - WARNING - <!DOCTYPE HTML PUBLIC "-//IETF//DTD HTML 2.0//EN">
<html><head>
<title>502 Proxy Error</title>
</head><body>
<h1>Proxy Error</h1>
<p>The proxy server received an invalid
response from an upstream server.<br />
The proxy server could not handle the request<p>Reason: <strong>DNS lookup failure for: clp-chat</strong></p></p>
<hr>
<address>Apache/2.4.18 (Ubuntu) Server at jarvis.ling.uni-potsdam.de Port 443</address>
</body></html>, retrying in 10 seconds...


KeyboardInterrupt: 